# Building Fact table for the results and sprints silver tables

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
target_name = f"{catalog_name}.{gold_schema}.fact_session_results"
results_table = f"{catalog_name}.{silver_schema}.results"
sprints_table = f"{catalog_name}.{silver_schema}.sprints"

### Read the silver tables

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = (
    spark.table(results_table)
    .withColumn("session_type",F.lit("RACE"))
    .drop("race_name", "race_date", "ingestion_timestamp", "SourceFile"))

sprints_df = (
    spark.table(sprints_table)
    .withColumn("session_type",F.lit("SPRINT"))
    .drop("race_name", "race_date", "ingestion_timestamp", "SourceFile"))

### Join the drivers and nationality tables and rename the region column

In [0]:
fact_session_df = results_df.unionByName(sprints_df)

In [0]:
fact_final_df = (
    fact_session_df
    .withColumn("is_win",F.col("final_position")==1)
    .withColumn("is_podium",F.col("final_position").between(1,3))
    .withColumn("has_points",F.col("points")>0)
)

In [0]:
display(fact_final_df)

### Write the dataframe into the gold schema

In [0]:
(
    fact_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_name)
)

In [0]:
display(spark.table(target_name))